# Quantum Circuit Hands-on Lab: From Qubit Basics to Real Algorithms

**เป้าหมาย:** เรียน Quantum Computing แบบลงมือทำจริงด้วย Qiskit โดยค่อย ๆ เพิ่มระดับจาก circuit ง่าย ๆ → superposition → entanglement → phase → oracle → quantum algorithms

แนวทางของ Lab นี้:
1. **Predict** — เดาผลลัพธ์ก่อน run
2. **Build** — สร้าง circuit ด้วย Qiskit
3. **Run** — ทดลองจริง
4. **Explain** — อธิบายว่าทำไมผลลัพธ์จึงเป็นแบบนั้น
5. **Modify** — เปลี่ยน circuit แล้วสังเกตผล
6. **Challenge** — ทำโจทย์โดยไม่ดูเฉลยก่อน

> แนะนำ: อย่าเพิ่งอ่านเฉลยของ Challenge ให้ลองทำเองก่อน

## 0. Setup

ถ้าใช้ environment เดิมของคุณ ให้ติดตั้ง Qiskit และ simulator ก่อน

```bash
pip install qiskit qiskit-aer matplotlib
```

ถ้า Jupyter ไม่เห็น environment ของคุณ ให้ติดตั้ง kernel ใน environment นั้นด้วย:

```bash
pip install ipykernel
python -m ipykernel install --user --name quantum1 --display-name "Python (quantum1)"
```

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

simulator = AerSimulator()

print("Qiskit environment is ready!")

Qiskit environment is ready!


# Level 1 — Your First Quantum Circuit

## 1.1 สร้าง |0⟩

Qubit เริ่มต้นที่

$$|0\rangle = \begin{bmatrix}1\\0\end{bmatrix}$$

สร้าง 1 qubit และ 1 classical bit แล้ววัดผล

**ก่อน Run:** คุณคาดว่าจะได้ `0` กี่เปอร์เซ็นต์?

In [2]:
qc = QuantumCircuit(1, 1)
qc.measure(0, 0)

print(qc)

     ┌─┐
  q: ┤M├
     └╥┘
c: 1/═╩═
      0 


In [3]:
result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)
plt.show()

{'0': 1000}


### Checkpoint 1

ตอบด้วยตัวเอง:

- ทำไมผลลัพธ์จึงเป็น `0` เกือบทั้งหมด?  
ค่า default เป็น $$|0\rangle$$ 
- `shots=1000` หมายถึงอะไร?  
run simulate ไป 1000 ครั้ง 
- classical bit มีหน้าที่อะไร?  
บันทึกค่าที่วัดได้

## 1.2 ทดลอง X gate

Pauli-X ทำหน้าที่คล้าย NOT gate:

$$X|0\rangle = |1\rangle$$

**ก่อน Run:** คาดการณ์ผลลัพธ์ก่อน

In [4]:
qc = QuantumCircuit(1, 1)
qc.x(0)
qc.measure(0, 0)

print(qc)

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

plot_histogram(counts)
plt.show()

     ┌───┐┌─┐
  q: ┤ X ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 
{'1': 1000}


### Challenge 1

สร้าง circuit ที่ทำให้ผลสุดท้ายเป็น `0` โดยใช้ X gate **สองครั้ง**

Hint:

$$X(X|0\rangle)=|0\rangle$$

# Level 2 — Superposition

ตอนนี้เราจะเริ่มเห็นสิ่งที่ต่างจาก classical computing

Hadamard gate:

$$H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}=|+\rangle$$

ดังนั้นเมื่อวัด จะได้

- `0` ≈ 50%
- `1` ≈ 50%

แต่ระวัง: **superposition ไม่ได้หมายความว่า qubit มีค่า 0 และ 1 แบบ classical พร้อมกันแล้วเราสามารถอ่านทั้งสองค่าได้** การ measurement จะให้ผลลัพธ์หนึ่งค่า

In [5]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

print(qc)

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

plot_histogram(counts)
plt.show()

     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 
{'1': 482, '0': 518}


## 2.1 เปลี่ยนจำนวน shots

ลองเปลี่ยนเป็น `10`, `100`, `10000`

**สังเกต:** ทำไม 10 shots อาจไม่ใกล้ 50/50 เท่า 10,000 shots?

In [6]:
for shots in [10, 100, 1000, 10000]:
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    result = simulator.run(qc, shots=shots).result()
    print(shots, result.get_counts())

10 {'1': 4, '0': 6}
100 {'0': 48, '1': 52}
1000 {'0': 511, '1': 489}
10000 {'1': 5031, '0': 4969}


## 2.2 H + H

จากสิ่งที่คุณเคยเรียน:

$$H^2=I$$

ดังนั้น

$$H(H|0\rangle)=|0\rangle$$

### Challenge 2

สร้าง circuit `H → H → measurement`

**ก่อน run:** ทายผลลัพธ์

จากนั้นทดลองและอธิบายว่าทำไมจึงกลับมาเป็น `0`

# Level 3 — Phase and Interference

นี่เป็นจุดสำคัญมาก เพราะ quantum computing ไม่ได้อาศัยแค่ probability แต่ใช้ **amplitude และ phase**

สร้าง state:

$$|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}$$

แล้วใช้ Z:

$$Z|+\rangle = | - \rangle
=\frac{|0\rangle-|1\rangle}{\sqrt2}$$

ถ้า measurement ทันทีหลัง Z เราอาจยังเห็น 50/50

แต่ถ้าใช้ H ต่อ:

$$H|+\rangle=|0\rangle$$

$$H|-\rangle=|1\rangle$$

นี่คือ interference ที่เราจะใช้ใน algorithms ต่อไป

In [7]:
# H -> Z -> H
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.z(0)
qc.h(0)
qc.measure(0, 0)

print(qc)

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

plot_histogram(counts)
plt.show()

     ┌───┐┌───┐┌───┐┌─┐
  q: ┤ H ├┤ Z ├┤ H ├┤M├
     └───┘└───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 
{'1': 1000}


### Challenge 3

ทดลอง circuit ต่อไปนี้ทีละแบบ:

1. `H`
2. `H → Z`
3. `H → Z → H`
4. `H → X → H`

ทายผลลัพธ์ก่อนทุกครั้ง

คำถามสำคัญ:

> ทำไมบาง gate เปลี่ยน measurement probability และบาง gate เหมือน "ไม่ทำอะไร" จนกระทั่งเราใช้ gate อื่นตามมา?

# Level 4 — Two Qubits and Entanglement

สร้าง Bell state:

$$|\Phi^+\rangle =
\frac{|00\rangle+|11\rangle}{\sqrt2}$$

ขั้นตอน:

1. H ที่ qubit 0
2. CNOT โดย qubit 0 เป็น control และ qubit 1 เป็น target
3. measurement

ผลควรเป็นเพียง `00` และ `11`

In [8]:
qc = QuantumCircuit(2, 2)

qc.h(0)
qc.cx(0, 1)

qc.measure([0, 1], [0, 1])

print(qc)

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

plot_histogram(counts)
plt.show()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'11': 491, '00': 509}


### Think Before You Answer

ถ้า measurement ให้ `00` แสดงว่าอะไร?

ถ้าให้ `11` แสดงว่าอะไร?

แล้วทำไม `01` และ `10` จึงไม่เกิดใน ideal simulator?

> จุดสำคัญ: correlation นี้ไม่ใช่แค่ "สุ่มเลขเดียวกันสองครั้ง" แต่เกิดจาก joint quantum state

## 4.1 Bell State Variations

ลองสร้าง:

- $\Phi^+ = (|00\rangle+|11\rangle)/\sqrt2$
- $\Phi^- = (|00\rangle-|11\rangle)/\sqrt2$
- $\Psi^+ = (|01\rangle+|10\rangle)/\sqrt2$
- $\Psi^- = (|01\rangle-|10\rangle)/\sqrt2$

### Challenge 4

หาว่า gate sequence ใดสร้างแต่ละ state

Hint: เริ่มจาก `|00⟩` และใช้ H, X, Z, CX

# Level 5 — Measurement vs Statevector

จนถึงตอนนี้เราใช้ measurement เป็นหลัก

แต่ตอนเรียน quantum mechanics เราสนใจ state:

$$|\psi\rangle = \sum_x \alpha_x|x\rangle$$

ลองใช้ statevector simulator เพื่อดู amplitudes โดยตรง

In [9]:
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(1)
qc.h(0)

state = Statevector.from_instruction(qc)

print(state)
print("Probabilities:", state.probabilities_dict())

Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))
Probabilities: {np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}


### Experiment

เปรียบเทียบ statevector ของ:

1. `|0⟩`
2. `X|0⟩`
3. `H|0⟩`
4. `HZ|+⟩`
5. `HZH|0⟩`

ดูทั้ง amplitude และ probability

**เป้าหมาย:** แยกให้ออกว่า

- amplitude
- probability
- phase

คือคนละสิ่งกัน

# Level 6 — Build a Quantum Function

เราจะสร้าง circuit ที่ทำหน้าที่เหมือน function

$$f(x)$$

โดยใช้ oracle

สำหรับตัวอย่างง่าย:

$$f(x)=x$$

เมื่อ input เป็น `|0⟩` output = 0

เมื่อ input เป็น `|1⟩` output = 1

### Challenge 5

สร้าง circuit ที่:

- รับ input จาก qubit 0
- คำนวณ $f(x)=x$
- เก็บ output ใน qubit 1

Hint: CNOT เหมาะมาก

In [17]:
qc = QuantumCircuit(2,2)

# q0 = input
# q1 = output (เริ่มต้นเป็น 0)

# คำนวณ f(x) = x
qc.x(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

# แสดง circuit
print(qc.draw())

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)

     ┌───┐     ┌─┐   
q_0: ┤ X ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'11': 1000}


# Level 7 — Deutsch Algorithm

ตอนนี้เริ่มเข้าสู่ **quantum algorithm จริง**

Deutsch algorithm ต้องการตอบว่า function แบบ 1-bit:

$$f:\{0,1\}\rightarrow\{0,1\}$$

เป็น

- **constant:** $f(0)=f(1)$
- **balanced:** $f(0)\neq f(1)$

โดย classical ต้อง query function หลายกรณี แต่ quantum algorithm ใช้ oracle และ interference เพื่อแยกประเภทได้ด้วยการ query oracle หนึ่งครั้ง

วงจรโดยแนวคิด:

```text
q0: ──H──────Uf────H──M
q1: ──X──────Uf───────
```

### Tasks

1. สร้าง oracle สำหรับ `f(x)=0`
2. สร้าง oracle สำหรับ `f(x)=1`
3. สร้าง oracle สำหรับ `f(x)=x`
4. สร้าง oracle สำหรับ `f(x)=NOT x`
5. รัน Deutsch algorithm
6. อธิบายว่า measurement `0` และ `1` บอกอะไร

In [ ]:
# Deutsch algorithm starter
# Try to implement the oracle first.

def deutsch_oracle(qc, function):
    # function: 0, 1, 2, 3
    # 0 -> f(x)=0
    # 1 -> f(x)=1
    # 2 -> f(x)=x
    # 3 -> f(x)=NOT x

    if function == 0:
        pass
    elif function == 1:
        qc.x(1)
    elif function == 2:
        qc.cx(0, 1)
    elif function == 3:
        qc.x(0)
        qc.cx(0, 1)
        qc.x(0)

# Your implementation

# Level 8 — Grover Search

Grover's algorithm คือหนึ่งใน algorithms ที่สำคัญที่สุดสำหรับ quantum search

สำหรับ search space ขนาด $N$:

Classical:
$$O(N)$$

Grover:
$$O(\sqrt N)$$

ใน Lab นี้เริ่มจาก 2 qubits → 4 possible states:

$$|00\rangle, |01\rangle, |10\rangle, |11\rangle$$

สมมติ target คือ `|10⟩`

แนวคิด:

1. Prepare uniform superposition
2. Oracle ทำ phase flip ให้ target
3. Diffusion operator เพิ่ม amplitude ของ target
4. Measure

### Challenge 6

Implement Grover สำหรับ 2 qubits โดย target = `10`.

ก่อน run ให้ตอบ:

> หลัง Grover แล้ว state ไหนควรมี probability สูงที่สุด?

In [ ]:
# Grover 2-qubit starter

qc = QuantumCircuit(2, 2)

# 1. Superposition
qc.h([0, 1])

# 2. Oracle for target |10>
# You need to implement this.

# 3. Diffusion operator
# You need to implement this.

qc.measure([0, 1], [0, 1])

print(qc)

# Level 9 — Deutsch-Jozsa Algorithm

ขยายแนวคิดจาก Deutsch ไปเป็นหลาย qubits

Problem:

มี function

$$f:\{0,1\}^n\rightarrow\{0,1\}$$

และเรารู้ว่า function เป็น:

- constant
- balanced

Goal: determine which one

### Challenge 7

สร้าง Deutsch-Jozsa สำหรับ $n=2$

ทดลองอย่างน้อย:

1. constant-0 oracle
2. constant-1 oracle
3. balanced oracle เช่น $f(x_0,x_1)=x_0$
4. balanced oracle เช่น $f(x_0,x_1)=x_0\oplus x_1$

เปรียบเทียบ measurement results

# Level 10 — Mini Project: Build Your Own Quantum Algorithm Experiment

ตอนนี้หยุดตาม tutorial แล้วออกแบบ experiment เอง

เลือก **หนึ่ง**:

### Option A — Bernstein-Vazirani
ค้นหา hidden bit string $s$

$$f(x)=s\cdot x \pmod 2$$

เป้าหมาย: recover `s`

### Option B — Grover
สร้าง search problem 3 qubits และหา target state

### Option C — Quantum Teleportation
สร้าง protocol:

Alice → entanglement → measurement → classical communication → Bob

### Option D — Superdense Coding
ส่ง classical information 2 bits โดยใช้ qubit 1 qubit + shared entanglement

### Option E — VQE mini experiment
สร้าง Hamiltonian ง่าย ๆ และใช้ parameterized circuit เพื่อประมาณ ground-state energy

สำหรับ capstone Quantum ML แนะนำให้ลอง **VQE** หลังจากทำพื้นฐานทั้งหมด

# Final Challenge — Explain, Don't Just Code

ห้ามเปิด code แล้วตอบจากความจำ

อธิบายด้วยภาษาของตัวเอง:

1. Qubit ต่างจาก classical bit อย่างไร?
2. Superposition คืออะไร?
3. Measurement ทำอะไรกับ state?
4. Amplitude กับ probability ต่างกันอย่างไร?
5. Phase สำคัญอย่างไร?
6. Entanglement คืออะไร?
7. ทำไม HZH ถึงเกี่ยวข้องกับ X?
8. Oracle ใน quantum algorithm คืออะไร?
9. Diffusion operator ของ Grover ทำอะไร?
10. Deutsch-Jozsa ใช้ interference เพื่อช่วยแก้ปัญหาอย่างไร?
11. ทำไม Grover ถึงมี quadratic speedup?
12. Quantum algorithm แตกต่างจาก "สุ่มหลายคำตอบแล้วเลือก" อย่างไร?

ถ้าตอบไม่ได้ ให้ย้อนกลับไปทำ experiment ที่เกี่ยวข้องอีกครั้ง

# Suggested Learning Path

| Level | Topic | Skill |
|---|---|---|
| 1 | Qubit / X / Measurement | Circuit syntax |
| 2 | H / Superposition | Probability |
| 3 | Z / Phase / HZH | Interference |
| 4 | CX / Bell state | Entanglement |
| 5 | Statevector | Quantum state |
| 6 | Oracle | Quantum function |
| 7 | Deutsch | First algorithm |
| 8 | Grover | Search algorithm |
| 9 | Deutsch-Jozsa | Multi-qubit algorithm |
| 10 | BV / Teleportation / VQE | Independent implementation |

## Rule for mastering each level

อย่าไป Level ถัดไปจนกว่าจะทำได้ 3 อย่าง:

**1. Predict** — ทายผลก่อน run

**2. Implement** — เขียน circuit เองโดยไม่ copy

**3. Explain** — อธิบายได้ว่า gate แต่ละตัวเปลี่ยน state อย่างไร

จากนั้นค่อยไป QML เช่น:

`Quantum circuit → Parameterized circuit → Expectation value → VQE → Quantum Kernel → VQC/QNN`